# Workshop 2 — Machine Learning (Unidad 2)
## Regresión — Flight Price Prediction

**Integrantes:**
- Juan José Escobar Saldarriaga
- Jeremías Figueroa García
- David Quintero Gallego

**Fecha:** 2026-09-01

---

Pipelines, entrenamiento, comparación de modelos y validación cruzada. Este notebook resuelve el problema de **regresión** usando el dataset **Flight Price Prediction**.

Dataset: https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction

**Nota:** el workshop completo consta de 2 notebooks — este (regresión) y su contraparte de clasificación. El punto 28 (conclusiones, Fase 8) pide comparar ambos problemas; esa comparación debe ser consistente entre los dos notebooks.

Cada punto requiere código **y** una celda de justificación en markdown explicando la decisión tomada — no basta con ejecutar el código.

---

# Fase 0 — Importación de librerías y datos

**En caso de trabajar con Google Colab, es necesario descargar el dataset en Drive y conectar el notebook con drive:**

In [ ]:
from google.colab import drive
import sys
drive.mount('/content/drive')

%cd '/content/drive/MyDrive/Colab Notebooks/data'
#puede confirmar su ubicación mediante:
%ls

## Librerías

Importa aquí todas las librerías que vas a necesitar. Como mínimo:
- `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`
- De `sklearn.model_selection`: `train_test_split`, `cross_val_score` o `cross_validate`
- De `sklearn.compose`: `ColumnTransformer`
- De `sklearn.pipeline`: `Pipeline`
- De `sklearn.preprocessing`: `StandardScaler`, `MinMaxScaler`, `RobustScaler`, `OneHotEncoder`, `OrdinalEncoder`
- De `sklearn.impute`: `SimpleImputer`
- Modelos: `LinearRegression`, `KNeighborsRegressor`, `DecisionTreeRegressor`, `RandomForestRegressor`, `GradientBoostingRegressor`
- Métricas: `mean_absolute_error`, `mean_squared_error`, `r2_score`

In [2]:
# Importa aquí todas las librerías necesarias:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Define random_state para reproducibilidad:
random_state = 42

---

# Fase 1 — EDA, limpieza y preprocesamiento

## 1. Descripcion del problema y variable objetivo

El dataset agrupa opciones de reserva de vuelos publicadas en **Ease My Trip**, una plataforma en linea de venta de tiquetes aereos que los pasajeros usan directamente para comprar sus vuelos. El objetivo declarado por los autores del dataset es analizar estos datos y aplicar pruebas estadisticas para extraer informacion util, entrenando un modelo de **regresion lineal** que prediga una variable continua. Es decir, este es un problema de **regresion**, y la variable objetivo (target) es `price` (el precio del tiquete).

El estudio original plantea 5 preguntas de investigacion que sirven de guia para el analisis exploratorio:

a) ¿El precio varia segun la aerolinea?
b) ¿Como afecta el precio comprar el tiquete con 1 o 2 dias de anticipacion?
c) ¿El precio cambia segun la hora de salida y de llegada?
d) ¿Como cambia el precio segun la ciudad de origen y de destino?
e) ¿Como varia el precio entre clase economica y clase Business?

**Recoleccion de datos:** la informacion se extrajo del sitio web de Ease My Trip con la herramienta de scraping Octoparse, durante 50 dias (11 de febrero al 31 de marzo de 2022), en dos tandas separadas: una para tiquetes de clase economica y otra para clase Business. Los autores reportan haber extraido 300261 registros en total (mas adelante, en el punto 2, se compara esta cifra con el tamano real del dataset).

**Alcance:** el dataset cubre unicamente vuelos domesticos entre las 6 principales ciudades metropolitanas de India; no incluye vuelos internacionales ni de otros paises.

## 2. Carga del dataset y diccionario de datos

Cargue el dataset con pandas, muestre `head()`/`tail()`, `.shape` y `.info()`. Construya el diccionario de datos (columna, tipo esperado, significado).

In [3]:
# Carga del dataset:
df = pd.read_csv('data/Flights.csv')

In [6]:
# head() / tail():
df.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


In [7]:
df.tail()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
300148,300148,Vistara,UK-822,Chennai,Morning,one,Evening,Hyderabad,Business,10.08,49,69265
300149,300149,Vistara,UK-826,Chennai,Afternoon,one,Night,Hyderabad,Business,10.42,49,77105
300150,300150,Vistara,UK-832,Chennai,Early_Morning,one,Night,Hyderabad,Business,13.83,49,79099
300151,300151,Vistara,UK-828,Chennai,Early_Morning,one,Evening,Hyderabad,Business,10.00,49,81585
300152,300152,Vistara,UK-822,Chennai,Morning,one,Evening,Hyderabad,Business,10.08,49,81585


In [8]:
# .shape y .info():
df.shape

(300153, 12)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        300153 non-null  int64  
 1   airline           300153 non-null  str    
 2   flight            300153 non-null  str    
 3   source_city       300153 non-null  str    
 4   departure_time    300153 non-null  str    
 5   stops             300153 non-null  str    
 6   arrival_time      300153 non-null  str    
 7   destination_city  300153 non-null  str    
 8   class             300153 non-null  str    
 9   duration          300153 non-null  float64
 10  days_left         300153 non-null  int64  
 11  price             300153 non-null  int64  
dtypes: float64(1), int64(3), str(8)
memory usage: 27.5 MB


In [10]:
df.describe()

,Unnamed: 0,duration,days_left,price
count,300153.000000,300153.000000,300153.000000,300153.000000
mean,150076.000000,12.221021,26.004751,20889.660523
std,86646.852011,7.191997,13.561004,22697.767366
min,0.000000,0.830000,1.000000,1105.000000
25%,75038.000000,6.830000,15.000000,4783.000000
50%,150076.000000,11.250000,26.000000,7425.000000
75%,225114.000000,16.170000,38.000000,42521.000000
max,300152.000000,49.830000,49.000000,123071.000000


**Filas:** `300153`
**Columnas:** `12`

**Diccionario de datos:**

| Columna | Tipo esperado | Significado |
|---|---|---|
| `Unnamed: 0` | Numero entero (int64) | Indice/identificador secuencial heredado, no aporta informacion del vuelo |
| `airline` | Categorica (str) | Aerolinea que opera el vuelo (6 categorias) |
| `flight` | Categorica (str) | Codigo del vuelo/avion (alta cardinalidad, 1561 valores unicos) |
| `source_city` | Categorica (str) | Ciudad de origen del vuelo (6 categorias) |
| `departure_time` | Categorica (str) | Franja horaria de salida, agrupada en 6 bloques (Morning, Evening, etc.) |
| `stops` | Categorica (str) | Numero de escalas entre origen y destino (zero, one, two_or_more) |
| `arrival_time` | Categorica (str) | Franja horaria de llegada, agrupada en 6 bloques |
| `destination_city` | Categorica (str) | Ciudad de destino del vuelo (6 categorias) |
| `class` | Categorica (str) | Clase del tiquete (Economy o Business) |
| `duration` | Numerico continuo (float64) | Duracion total del vuelo, en horas |
| `days_left` | Numerico entero (int64) | Dias de anticipacion entre la fecha de compra y la fecha del vuelo |
| `price` | Numerico entero (int64) | Variable objetivo: precio del tiquete |


**Observaciones iniciales:**

* Varias columnas categoricas en realidad podrian tratarse de forma numerica directamente: por ejemplo, `stops` guarda "zero", "one" y "two_or_more" como texto, pudiendo representarse simplemente como 0, 1 y 2 (tiene un orden natural). Esto se retomara en el punto 9 (codificacion de categoricas).
* La columna `Unnamed: 0` es casi con certeza un identificador secuencial heredado del proceso de extraccion/guardado del dataset, sin informacion util sobre el vuelo; se eliminara mas adelante en la limpieza. Aun asi, es una buena senal de integridad: sus valores van de 0 a 300152 sin saltos, lo que sugiere que el dataset esta completo y no se perdieron filas al guardarlo.
* Dato curioso: la documentacion de Kaggle menciona 300261 registros extraidos, pero el dataset real tiene 300153 filas, 108 menos. Es posible que quien publico el dataset haya limpiado algunos registros (duplicados o incompletos) antes de subirlo, o que la cifra de la documentacion incluya registros descartados durante el scraping que nunca llegaron al CSV final. De cualquier forma, 300153 filas siguen siendo mas que suficientes para entrenar el modelo, asi que esta diferencia no representa un problema.

## 3. Estadísticos descriptivos

Use `.describe()` para las columnas numéricas. Interprete al menos 3 estadísticos en términos del problema.

In [12]:
# .describe():
df.describe()

,Unnamed: 0,duration,days_left,price
count,300153.000000,300153.000000,300153.000000,300153.000000
mean,150076.000000,12.221021,26.004751,20889.660523
std,86646.852011,7.191997,13.561004,22697.767366
min,0.000000,0.830000,1.000000,1105.000000
25%,75038.000000,6.830000,15.000000,4783.000000
50%,150076.000000,11.250000,26.000000,7425.000000
75%,225114.000000,16.170000,38.000000,42521.000000
max,300152.000000,49.830000,49.000000,123071.000000


**Interpretacion:**

`Unnamed: 0` aparece en la tabla por ser numerica, pero se ignora en este analisis: ya se establecio que es un identificador sin informacion sobre el vuelo.

**`duration`** (horas): el minimo es 0.83 horas (menos de 1 hora), razonable para un vuelo directo entre ciudades cercanas sin escalas. La media es de 12.22 horas, un numero que llama la atencion tratandose de vuelos domesticos dentro de un mismo pais; probablemente refleja que buena parte de los vuelos incluyen escalas que alargan bastante el tiempo total. El maximo es de 49.83 horas, un valor bastante raro (mas de 2 dias entre origen y destino) que muy probablemente corresponda a un vuelo con varias escalas largas; esto se retomara en el punto de outliers (punto 7). La mediana (11.25) queda un poco por debajo de la media, señal de un sesgo leve hacia la derecha (se confirma mas adelante con el histograma). La desviacion estandar es de 7.19 horas, coherente con el rango entre percentiles (25%: 6.83, 75%: 16.17): hay bastante dispersion, posiblemente explicada tambien por la cantidad de escalas de cada vuelo.

**`days_left`** (dias de anticipacion): la media es de 26 dias y la mediana tambien de 26, practicamente sin sesgo (coherente con planear un viaje con cerca de un mes de anticipacion). El minimo es 1 dia (ya mencionado en la documentacion del dataset, que se enfoca en compras de ultimo momento) y el maximo es 49 dias, sin nada extraordinario: el 75% de los datos ya esta en 38 dias, y entre percentil y percentil hay una diferencia de 11 a 12 dias (25%: 15, 50%: 26, 75%: 38), asi que el maximo no se aleja de forma desproporcionada del resto. La desviacion estandar de 13.56 es coherente con ese rango de 1 a 49 dias: es una variable con comportamiento bastante "humano" y sin mayores sorpresas.

**`price`** (variable objetivo): aqui si aparece un sesgo fuerte a la derecha. La mediana esta en 7425 y la media en 20889.66, casi 3 veces mas alta, señal de que un grupo de vuelos muy costosos empuja el promedio muy por encima del precio "tipico". La desviacion estandar (22697.77) es incluso mayor que la propia media, algo que refuerza lo disperso que es el precio hacia los valores altos. El minimo es 1105 (un vuelo economico basico), y el salto entre percentiles es muy revelador: del 25% (4783) al 50% (7425) la diferencia es moderada, pero del 50% (7425) al 75% (42521) hay un salto enorme, lo que quiere decir que la mitad "cara" del dataset se dispara muy por encima de la mitad "barata". El maximo, 123071, es casi 3 veces el percentil 75, casi con certeza un vuelo Business de larga duracion (esto se confirmara mas adelante en el analisis, en vez de asumirlo aqui). Sobre la unidad: la documentacion no la especifica explicitamente, pero al tratarse de vuelos domesticos en India con Ease My Trip, lo mas probable es que sean rupias indias (INR); en dolares estos precios no tendrian sentido para vuelos domesticos.

Un dato adicional que ayuda a comparar objetivamente que tan dispersa es cada variable (ya que sus escalas son muy distintas) es el coeficiente de variacion (desviacion estandar / media):
* `duration`: 7.19 / 12.22 = 0.59
* `days_left`: 13.56 / 26.00 = 0.52
* `price`: 22697.77 / 20889.66 = 1.09

El de `price` es el doble que el de las otras dos variables, confirmando en numeros que el precio es, con diferencia, la variable mas volatil de las tres.

## 4. Valores nulos: identificación y estrategia

Identifique valores nulos por columna, calcule su porcentaje, y proponga una estrategia de imputación o eliminación. Justifique según el porcentaje de nulos y el tipo de variable.

In [ ]:
# Nulos por columna y porcentaje:


**Estrategia de tratamiento de nulos y justificación:** *(completar)*

In [ ]:
# Aplicación de la estrategia de nulos:


## 5. Duplicados

Identifique y elimine duplicados si existen. Reporte cuántos se eliminaron.

In [ ]:
# Identificación y eliminación de duplicados:


**Observación:** *(completar — cuántos se eliminaron y si tiene sentido que existieran)*

## 6. Inconsistencias de formato

Revise inconsistencias de formato en columnas categóricas o de texto (mayúsculas/minúsculas, espacios, etiquetas equivalentes) y corríjalas.

In [ ]:
# Revisión con .unique() de columnas categóricas/texto:


In [ ]:
# Corrección de inconsistencias encontradas:


**Justificación de las correcciones:** *(completar)*

## 7. Outliers (criterio IQR)

Identifique outliers en al menos dos columnas numéricas relevantes usando el criterio IQR. Repórtelos y decida si se tratan o se documentan sin eliminar, justificando la decisión (¿sentido de negocio o error de captura?).

In [ ]:
# IQR - columna numérica relevante 1:


In [ ]:
# IQR - columna numérica relevante 2:


**Decisión sobre los outliers y justificación:** *(completar)*

## 8. Gráficas de EDA

Para cada gráfica: código, imagen e interpretación con una recomendación como responsable de la toma de decisiones.

### 8a. Gráfico de barras

Distribución de una variable categórica relevante.

**Interpretación:** *(completar)*

**Recomendación:** *(completar)*

### 8b. Gráfico de pie

Proporción de una variable categórica binaria o con pocas categorías.

**Interpretación:** *(completar)*

**Recomendación:** *(completar)*

### 8c. Histograma

Distribución de una variable numérica relevante (considere el target).

**Interpretación:** *(completar)*

**Recomendación:** *(completar)*

### 8d. Scatter plot

Relación entre dos variables numéricas relevantes.

**Interpretación:** *(completar)*

**Recomendación:** *(completar)*

### 8e. Boxplot

Al menos una variable numérica; relaciónelo con los outliers del punto 7.

**Interpretación:** *(completar)*

**Recomendación:** *(completar)*

## 9. Codificación de variables categóricas

Para cada variable categórica, decida y justifique la técnica de codificación (One-Hot, Ordinal o Binary Encoding), considerando orden natural, cardinalidad, y los algoritmos a usar.

In [ ]:
# Revisión de cardinalidad de cada columna categórica:


**Decisión de encoding por columna y justificación:** *(completar)*

## 10. Escalamiento de variables numéricas

Decida y justifique si es necesario escalar las variables numéricas y con qué técnica (Min-Max, StandardScaler o RobustScaler), considerando los outliers del punto 7 y los algoritmos a entrenar.

**Decisión de escalamiento y justificación:** *(completar)*

---

# Fase 2 — División de datos y Pipelines

## 11. División Train / Validation / Test

Divida el dataset en train / validation / test (ej. 70/15/15). Explique con sus palabras el propósito de cada uno de los tres conjuntos y por qué no basta con solo train/test.

In [ ]:
# Separación de X e y, y split en train/validation/test:


**Explicación del propósito de cada conjunto:** *(completar)*

## 12. Pipeline con ColumnTransformer

Construya un Pipeline con ColumnTransformer que aplique el preprocesamiento numérico (imputación + escalamiento) y categórico (imputación + encoding) definidos en la Fase 1.

In [ ]:
# Identificación de columnas numéricas y categóricas:


In [ ]:
# Pipeline numérico, pipeline(s) categórico(s) y ColumnTransformer:


## 13. Verificación de que no hay fuga de datos (data leakage)

Confirme explícitamente que el `.fit()` de cada transformación se realizó solo sobre `X_train`, y que sobre `X_val`/`X_test` únicamente se aplicó `.transform()`.

**Verificación:** *(completar — explicar en qué parte del código se garantiza esto)*

---

# Fase 3 — Modelado: Regresión (Flight Price Prediction)

## 14. Entrenamiento de modelos

Entrene los siguientes modelos sobre el conjunto de entrenamiento: Regresión Lineal Múltiple, KNN Regressor, Decision Tree Regressor, Random Forest Regressor y Gradient Boosting Regressor.

In [ ]:
# Pipeline + entrenamiento: Regresión Lineal Múltiple


In [ ]:
# Pipeline + entrenamiento: KNN Regressor


In [ ]:
# Pipeline + entrenamiento: Decision Tree Regressor


In [ ]:
# Pipeline + entrenamiento: Random Forest Regressor


In [ ]:
# Pipeline + entrenamiento: Gradient Boosting Regressor


## 15. Predicciones sobre validación

Para cada modelo, genere las predicciones sobre el conjunto de validación.

In [ ]:
# Predicciones de cada modelo sobre X_val:


---

# Fase 5 — Métricas en train y validación: detección de overfitting/underfitting

## 18. Métricas: MAE, MSE y R²

Para cada uno de los 5 modelos, calcule MAE, MSE y R² tanto sobre train como sobre validación (dos evaluaciones por modelo). Interprete cada métrica en el contexto de precios de vuelos.

In [ ]:
# Función de evaluación y cálculo de métricas en train y val para cada modelo:


**Interpretación (overfitting/underfitting y significado de las métricas):** *(completar)*

## 20. Tabla comparativa (train y validación)

Construya una tabla comparativa que incluya train y validación para todos los modelos.

In [ ]:
# Tabla comparativa train/val:


## 21. Selección del mejor modelo

Seleccione el mejor modelo y justifique la elección — discuta trade-offs (tiempo de entrenamiento, interpretabilidad, señales de over/underfitting), no solo la métrica más alta.

**Justificación de la selección:** *(completar)*

---

# Fase 6 — Evaluación final en el conjunto de Test

## 22. Evaluación de todos los modelos en test

Evalúe TODOS los modelos entrenados sobre el conjunto de test, usando las mismas métricas de la Fase 5.

> ⚠️ El conjunto de test debe permanecer sin tocar hasta este punto.

In [ ]:
# Evaluación de todos los modelos sobre X_test:


## 23. Tabla comparativa final (test) vs. validación

Construya la tabla final con los resultados en test y compárela contra la tabla de validación de la Fase 5: ¿se mantiene el ranking? ¿hay diferencias importantes?

In [ ]:
# Tabla comparativa final en test:


**Comparación val vs. test e interpretación:** *(completar)*

---

# Fase 7 — Cross Validation

## 24. ¿Qué es K-Fold Cross Validation?

Investigue y explique con sus propias palabras qué es la validación cruzada, cómo funciona, y por qué puede dar una estimación más confiable que un único split de train/validation.

**Explicación:** *(completar)*

## 25. K-Fold Cross Validation sobre el mejor modelo

Implemente K-Fold Cross Validation (ej. `cross_val_score` o `cross_validate`) sobre el mejor modelo encontrado para este problema, usando las métricas relevantes.

In [ ]:
# Cross validation sobre el mejor modelo:


## 27. Comparación CV vs. validación original

Compare los resultados de cross validation contra el desempeño obtenido en el conjunto de validación de la Fase 5: ¿mejora, empeora o se mantiene similar? ¿La desviación estándar entre folds sugiere estabilidad o sensibilidad a los datos?

**Interpretación:** *(completar)*

---

# Fase 8 — Conclusiones

## 28. Conclusiones generales

Redacte un párrafo de conclusiones generales **comparando ambos problemas** (regresión y clasificación): ¿qué modelo funcionó mejor en cada caso y por qué, relacionándolo con las características de cada algoritmo (sesgo/varianza, sensibilidad a escala, manejo de categóricas, etc.)? ¿Qué limpieza fue más determinante? ¿Qué limitaciones tuvo el estudio y qué preguntas quedan abiertas?

Incluya un subplot (2,2) con las gráficas que considere más importantes para sustentar los hallazgos de **este** dataset (Flight Price Prediction).

*(Completar conclusiones — recuerde que esta sección debe ser consistente con la del notebook de clasificación)*

In [ ]:
# Subplot (2,2) con las gráficas más importantes de este dataset:
